In [4]:
#SETUP DOCKER+LANGFUSE

#!cd ~/Desktop/Mini-Project-LangFuse/langfuse
#!docker compose up -d
#http://localhost:3000/

In [5]:
#SETUP LLM+LANGFUSE

#Import keys
import os
from dotenv import load_dotenv
load_dotenv()

#Set up the LLM client
from google import genai
from google.genai import types
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

#Connect LangFuse
from langfuse import observe, Langfuse, get_client
langfuse = Langfuse()

In [6]:
#TOOLS

def check_drug_interactions(drug_a: str, drug_b: str) -> str:
    """Check if two drugs have any interactions or risks when taken together.
    Args:
        drug_a: First drug name.
        drug_b: Second drug name.
    """
    
    interactions = {
        ("ibuprofen", "lisinopril"): "WARNING: Ibuprofen may reduce the blood pressure lowering effect of lisinopril and increase risk of kidney damage.",
        ("aspirin", "warfarin"): "DANGER: Combining aspirin with warfarin significantly increases bleeding risk.",
        ("paracetamol", "ibuprofen"): "SAFE: These can generally be taken together at recommended doses.",
        ("metformin", "alcohol"): "WARNING: Alcohol can increase the risk of lactic acidosis with metformin.",
    }
    
    key = tuple(sorted([drug_a.lower(), drug_b.lower()]))
    
    return interactions.get(key, f"No known interaction data found for {drug_a} and {drug_b}.")

def check_symptoms(symptoms: str) -> str:
    """Look up possible conditions based on symptoms the user describes.
    Args:
        symptoms: The symptoms to look up.
    """
    
    symptom_map = {
        "chest pain": "Possible conditions: angina, heart attack, acid reflux, muscle strain. URGENT: If accompanied by shortness of breath, seek emergency care immediately.",
        "headache": "Possible conditions: tension headache, migraine, dehydration, sinusitis. Seek care if severe, sudden onset, or with fever/stiff neck.",
        "fever": "Possible conditions: viral infection, bacterial infection, flu. Seek care if above 39.5°C or lasting more than 3 days.",
    }
    
    for key, value in symptom_map.items():
        if key in symptoms.lower():
            return value
    
    return f"No data found for symptoms: {symptoms}. Please consult a healthcare professional."

def get_medication_info(medication: str) -> str:
    """Get basic information about a medication including uses, side effects, and dosage.
    Args:
        medication: The medication name.
    """
    
    meds = {
        "ibuprofen": "Type: NSAID. Used for: pain, inflammation, fever. Common side effects: stomach pain, nausea, dizziness. Max daily dose: 1200mg (OTC).",
        "paracetamol": "Type: Analgesic. Used for: pain, fever. Common side effects: rare at normal doses. Max daily dose: 4000mg. WARNING: liver damage risk if exceeded.",
        "lisinopril": "Type: ACE inhibitor. Used for: high blood pressure, heart failure. Common side effects: dry cough, dizziness, headache.",
        "metformin": "Type: Biguanide. Used for: type 2 diabetes. Common side effects: nausea, diarrhea, stomach pain.",
    }
    
    return meds.get(medication.lower(), f"No data found for {medication}.")

def check_emergency(situation: str) -> str:
    """Check if a situation requires emergency medical attention and provide emergency protocols.
    Args:
        situation: Description of the emergency situation.
    """
    
    emergencies = {
        "choking": "EMERGENCY: 1) Call emergency services. 2) For conscious adult: perform Heimlich maneuver. 3) For unconscious: begin CPR.",
        "heart attack": "EMERGENCY: 1) Call emergency services immediately. 2) Have patient chew aspirin (if not allergic). 3) Keep patient calm and seated. 4) Be ready to perform CPR.",
        "stroke": "EMERGENCY: Use FAST method - Face drooping, Arm weakness, Speech difficulty, Time to call emergency services.",
        "allergic reaction": "EMERGENCY: 1) Call emergency services. 2) Use epinephrine auto-injector if available. 3) Have patient lie down with legs elevated.",
        "bleeding": "EMERGENCY: 1) Apply direct pressure with clean cloth. 2) Keep pressure for 10-15 minutes. 3) If severe, call emergency services.",
    }
    
    for key, value in emergencies.items():
        if key in situation.lower():
            return value
    
    return f"No specific emergency protocol found for: {situation}. If in doubt, call emergency services immediately."

def get_patient_history(patient_id: str) -> str:
    """Look up a patient's medical history including conditions, medications, and allergies.
    Args:
        patient_id: The patient's ID number.
    """
    
    patients = {
        "P001": "Name: John Smith. Age: 65. Conditions: hypertension, type 2 diabetes. Current medications: lisinopril 10mg, metformin 500mg. Allergies: penicillin.",
        "P002": "Name: Maria Garcia. Age: 42. Conditions: asthma, anxiety. Current medications: salbutamol inhaler, sertraline 50mg. Allergies: aspirin, sulfa drugs.",
        "P003": "Name: James Wilson. Age: 55. Conditions: high cholesterol, arthritis. Current medications: atorvastatin 20mg, ibuprofen 400mg. Allergies: none known.",
    }
    
    return patients.get(patient_id.upper(), f"No patient found with ID: {patient_id}")

def book_appointment(department: str, urgency: str) -> str:
    """Book a medical appointment in a specific department.
    Args:
        department: Medical department (e.g. cardiology, general, emergency, neurology).
        urgency: How urgent the appointment is (routine, urgent, emergency).
    """
    
    if urgency.lower() == "emergency":
        return f"EMERGENCY referral created for {department}. Patient should go to Emergency Department immediately."
    elif urgency.lower() == "urgent":
        return f"Urgent appointment booked in {department} for tomorrow morning."
    else:
        return f"Routine appointment booked in {department} for next week."

#Mapping
tool_functions = {
    "check_drug_interactions": check_drug_interactions,
    "check_symptoms": check_symptoms,
    "get_medication_info": get_medication_info,
    "check_emergency": check_emergency,
    "get_patient_history": get_patient_history,
    "book_appointment": book_appointment,
}

#Execute tool and trace it in LangFuse
@observe()
def execute_tool_traced(tool_name, tool_args):
    tool_func = tool_functions[tool_name]
    return tool_func(**tool_args)

In [13]:
#TRIAGE AGENT

TRIAGE_PROMPT = """You are a medical triage agent. Your ONLY job is to classify 
the user's question into one of these categories and respond with ONLY the category name:

- MEDICATION: Questions about drugs, side effects, interactions, dosage
- SYMPTOMS: Questions about symptoms, conditions, diagnosis
- EMERGENCY: Urgent situations, severe symptoms, life-threatening conditions
- GENERAL: General health questions, lifestyle, diet, exercise

Respond with ONLY one word: MEDICATION, SYMPTOMS, EMERGENCY, or GENERAL.
Nothing else."""

@observe()
def triage_agent(user_question):
    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=user_question,
        config=types.GenerateContentConfig(
            system_instruction=TRIAGE_PROMPT
        )
    )
    
    category = response.text.strip().upper()
    print(f"Triage → {category}")
    
    return category

In [8]:
#SPECIALISTS AGENTS

#PROMPTS
MEDICATION_PROMPT = """You are a medication specialist assistant. 
You are an expert on drugs, side effects, interactions, and dosage.
Use your tools to look up accurate medication information.
Always recommend consulting a doctor or pharmacist for personalized advice.
Never prescribe medication."""

SYMPTOMS_PROMPT = """You are a symptoms assessment assistant.
You help users understand their symptoms and possible conditions.
Use your tools to look up symptom information.
Always recommend seeing a doctor for proper diagnosis.
Never diagnose conditions."""

EMERGENCY_PROMPT = """You are an emergency medical assistant.
You handle urgent medical situations with clear, actionable instructions.
Use your tools to look up emergency protocols.
Always direct patients to call emergency services for life-threatening situations.
Be calm, clear, and direct."""

GENERAL_PROMPT = """You are a general health assistant.
You answer questions about lifestyle, diet, exercise, and general wellbeing.
If a question requires specific medical knowledge, recommend consulting a doctor.
Be helpful and encouraging."""

#AGENTS
@observe()
def medication_agent(user_question):
    tools = [check_drug_interactions, get_medication_info, get_patient_history]
    
    return _run_specialist(user_question, MEDICATION_PROMPT, tools)

@observe()
def symptoms_agent(user_question):
    tools = [check_symptoms, get_patient_history, book_appointment]
    
    return _run_specialist(user_question, SYMPTOMS_PROMPT, tools)

@observe()
def emergency_agent(user_question):
    tools = [check_emergency, get_patient_history, book_appointment]
    
    return _run_specialist(user_question, EMERGENCY_PROMPT, tools)

@observe()
def general_agent(user_question):
    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=user_question,
        config=types.GenerateContentConfig(
            system_instruction=GENERAL_PROMPT
        )
    )
    
    return response.text

In [9]:
#REACT LOOP FOR SPECIALIST AGENTS

def _run_specialist(user_question, system_prompt, tools, max_iterations=5):
    contents = [
        types.Content(role="user", parts=[types.Part(text=user_question)])
    ]
    
    #ReAct loop (Think → Act → Observe → Repeat)
    for i in range(max_iterations):
        
        #THINK: Ask Gemini what to do (answer directly or use a tool?)
        response = gemini_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,  
                tools=tools,                       
                automatic_function_calling=types.AutomaticFunctionCallingConfig(
                    disable=True
                )
            )
        )
        
        candidate = response.candidates[0]
        function_calls = response.function_calls
        
        #ACT + OBSERVE: If Gemini wants to use tool(s), execute them
        if function_calls:
            contents.append(candidate.content)
            for fc in function_calls:
                print(f"  TOOL: {fc.name}({fc.args})")
                result = execute_tool_traced(fc.name, fc.args)
                print(f"  RESULT: {result}")
                
                #Add tool result to conversation so Gemini can see it next iteration
                contents.append(
                    types.Content(
                        role="user",
                        parts=[types.Part.from_function_response(
                            name=fc.name,
                            response={"result": result}
                        )]
                    )
                )
        else:
            #No tool call → Gemini has enough info, return final answer
            return response.text
    
    return "Sorry, I couldn't find a complete answer."

In [10]:
#ORCHESTRATOR

@observe()
def multi_agent_healthcare(user_question):
    print(f"\n{'='*50}")
    print(f"User: {user_question}")
    print(f"{'='*50}")
    
    # Step 1: Triage
    category = triage_agent(user_question)
    
    # Step 2: Route to specialist
    if category == "MEDICATION":
        answer = medication_agent(user_question)
    elif category == "SYMPTOMS":
        answer = symptoms_agent(user_question)
    elif category == "EMERGENCY":
        answer = emergency_agent(user_question)
    else:
        answer = general_agent(user_question)
    
    print(f"\nFinal answer: {answer}")
    return answer

In [14]:
#TEST

#Test 1: Should route to MEDICATION agent
multi_agent_healthcare("Can I take ibuprofen with lisinopril?")

#Test 2: Should route to SYMPTOMS agent
multi_agent_healthcare("I've been having headaches for a week")

#Test 3: Should route to EMERGENCY agent
multi_agent_healthcare("Someone is choking and can't breathe")

#Test 4: Should route to GENERAL agent
multi_agent_healthcare("How much water should I drink per day?")

#Test 5: Complex — triage needs to decide
multi_agent_healthcare("Patient P001 wants to take aspirin but has been having chest pain")

get_client().flush()


User: Can I take ibuprofen with lisinopril?
Triage → MEDICATION
  TOOL: check_drug_interactions({'drug_b': 'lisinopril', 'drug_a': 'ibuprofen'})
  RESULT: WARNING: Ibuprofen may reduce the blood pressure lowering effect of lisinopril and increase risk of kidney damage.

Final answer: It is not recommended to take ibuprofen with lisinopril. Ibuprofen may reduce the blood pressure-lowering effect of lisinopril and increase the risk of kidney damage.

Please consult your doctor or pharmacist for personalized advice regarding your medications.

User: I've been having headaches for a week
Triage → SYMPTOMS
  TOOL: check_symptoms({'symptoms': 'headaches for a week'})
  RESULT: Possible conditions: tension headache, migraine, dehydration, sinusitis. Seek care if severe, sudden onset, or with fever/stiff neck.

Final answer: Headaches for a week could be due to several things like tension headaches, migraines, dehydration, or sinusitis. If your headaches are severe, came on suddenly, or are a


Final answer: I recommend calling emergency services immediately for chest pain. Do not take aspirin without medical advice.
